# CSSE4011 Machine Learning and Edge AI Workshop

### 0. Connect to a runtime and change the runtime type to **"T4 GPU"**

Click the down-arrow button in the top-right corner, as shown in the screenshots below.

> **Important:** Please make sure you select **T4 GPU**, as using a different runtime may lead to much longer execution times.

<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/change_runtime_type.png?raw=true" width="300"> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/runtime_type.png?raw=true" width="300">

## 1. Install Required Packages and Upload MNIST Data

In [ ]:
!git clone https://github.com/YangLi309/CSSE4011-ML-Workshop.git
%cd CSSE4011-ML-Workshop/

In [ ]:
# Install ONNX for model export
!pip3 install onnx onnxscript onnxruntime-gpu


## 2. Introduction

### What is PyTorch?
PyTorch is an open-source deep learning framework known for its flexibility and ease of use, making it ideal for rapid prototyping and research.

### What is ONNX?
ONNX (Open Neural Network Exchange) is an open standard format for representing machine learning models, enabling interoperability between various frameworks and devices.

### Why are these important for Edge AI?
- **PyTorch**: Facilitates easy model development and training.
- **ONNX**: Allows exporting models to a standardized format for deployment across different platforms, including resource-constrained edge devices.

### What You Will Learn in This Workshop
- Load and adapt a pretrained model for a new task.
- Evaluate model performance.
- Prune the model to improve inference time.
- Export the model to ONNX format.

## 3. Load Pretrained AlexNet and Adapte the Model for MNIST 10

In [5]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import alexnet, AlexNet_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device:", device)

# Pretrained AlexNet with the final layer replaced for MNIST (10 classes)
model = alexnet(weights=AlexNet_Weights.DEFAULT)
model.classifier[6] = nn.Linear(4096, 10)  # For 10 classes

model = model.to(device)
print("Pretrained AlexNet model is loaded.")

Current device: cuda
Pretrained AlexNet model is loaded.


## 4. Evaluate the Pretrained AlexNet on MNIST

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Transform for MNIST images (which are 28x28 grayscale)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load MNIST test data
val_dataset = ImageFolder(root='./data/val', transform=transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_dataset = ImageFolder(root='./data/test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Evaluate the model (with modified classifier but before fine-tuning)
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (before fine-tuning) on MNIST test data: {100 * correct / total:.2f}%")
print("Note: This accuracy is expected to be low as the model hasn't been finetuned for MNIST yet")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

Accuracy of AlexNet with modified classifier (before fine-tuning) on MNIST test data: 98.24%
Note: This accuracy is expected to be low as the model hasn't been finetuned for MNIST yet

Sample predictions:
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0


## 5. Freeze All Layers Except Classifier

<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/alexnet.png?raw=true" width="300">

In [8]:
# Transfer learning: keep ImageNet features fixed and only train the new MNIST head.
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

## 6. Load MNIST Train Dataset

In [9]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Assuming your dataset is stored in "./data/" with subfolders 0/, 1/, ..., 9/
train_dataset = ImageFolder(root='./data/train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

## 7. Finetune the Classifier on MNIST

### Training epochs and corresponding model accuracy

- **1 epoch:** 96.45% accuracy  
- **5 epochs:** 97.02% accuracy  
- **10 epochs:** 99.30% accuracy  

> **Note:** The default setting is **1 epoch** for time efficiency, but you may increase the number of epochs to achieve better performance.

> **Reference:** Training for **1 epoch** should take approximately **3 minutes**. If it takes significantly longer, check whether your runtime type is set to **CPU** instead of **GPU**.

In [10]:
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# You can increase the number of epochs if you want to have higher accuracy
num_epochs = 1

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Wrap train_loader with tqdm for progress bar
    train_loader_tqdm = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

    for images, labels in train_loader_tqdm:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        train_loader_tqdm.set_postfix({
            'loss': running_loss/total,
            'acc': 100.*correct/total
        })

    # Print epoch statistics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    print(f'\nEpoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

Epoch 1/1: 100%|██████████| 1875/1875 [01:09<00:00, 27.00it/s, loss=0.00359, acc=96.4]


Epoch 1/1 - Loss: 0.1148, Accuracy: 96.35%


## 8. Evaluate the Finetuned Model

In [11]:
# Evaluate the finetuned model on the test set (same loop structure as pre-training eval).
from tqdm import tqdm

correct = 0
total = 0
all_preds = []
all_labels = []
finetuned_model = model
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Evaluating'):
        images, labels = images.to(device), labels.to(device)
        outputs = finetuned_model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (after finetuning) on MNIST test data: {100 * correct / total:.2f}%")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

Evaluating: 100%|██████████| 313/313 [00:09<00:00, 32.32it/s]

Accuracy of AlexNet with modified classifier (after finetuning) on MNIST test data: 98.29%

Sample predictions:
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0


## 9. Model Export to ONNX
### 

In [ ]:
import torch
import onnx

# Paths
onnx_path = "../alexnet_mnist.onnx"

# Export on CPU for simplicity and compatibility
export_model = finetuned_model.cpu().eval()

# Example input for export
# Keep this shape consistent with your model's real expected input
dummy_input = (torch.randn(1, 3, 224, 224),)

# Export PyTorch model to ONNX
# dynamic_shapes makes batch dimension flexible
onnx_program = torch.onnx.export(
    export_model,
    dummy_input,
    dynamo=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_shapes={"x": {0: "batch"}},   # use your forward arg name, likely x
)

# Save ONNX file
onnx_program.save(onnx_path)
print(f"Saved ONNX model to: {onnx_path}")

# Optional sanity check
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model check passed.")

In [ ]:
import os
import onnx
from onnxconverter_common import float16

onnx_path = "../alexnet_mnist.onnx"
fp16_onnx_path = "../alexnet_mnist_fp16.onnx"

# Load FP32 ONNX model
model = onnx.load(onnx_path)

# Convert to FP16
# keep_io_types=False means model inputs/outputs will also become float16
# disable_shape_infer=True can help if shape inference is fragile on your graph
model_fp16 = float16.convert_float_to_float16(
    model,
    keep_io_types=False,
    disable_shape_infer=True,
)

# Save FP16 ONNX model
onnx.save(model_fp16, fp16_onnx_path)

print(f"Saved FP16 ONNX model to: {fp16_onnx_path}")

fp32_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
fp16_size_mb = os.path.getsize(fp16_onnx_path) / (1024 * 1024)

print(f"FP32 ONNX size: {fp32_size_mb:.2f} MB")
print(f"FP16 ONNX size: {fp16_size_mb:.2f} MB")
print(f"Size reduction: {(fp32_size_mb - fp16_size_mb) / fp32_size_mb * 100:.2f}%")

In [ ]:
import os
import numpy as np
import onnx
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

from onnxruntime.quantization import (
    quant_pre_process,
    quantize_static,
    CalibrationDataReader,
    QuantFormat,
    QuantType,
)

# Paths
fp32_onnx_path = "../alexnet_mnist.onnx"
preprocessed_onnx_path = "../alexnet_mnist_preprocessed.onnx"
int8_onnx_path = "../alexnet_mnist_int8_static.onnx"

# Calibration dataset root
calib_data_dir = "./data/train"

# Use a small subset for calibration to keep it fast
calib_subset_size = 512
calib_batch_size = 1


def build_calib_loader(data_root: str, batch_size: int = 1, subset_size: int | None = None):
    # Match the same preprocessing used for model inference
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    dataset = ImageFolder(root=data_root, transform=transform)

    if subset_size is not None:
        subset_size = min(subset_size, len(dataset))
        dataset = Subset(dataset, range(subset_size))

    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)


class ImageCalibrationDataReader(CalibrationDataReader):
    """
    ONNX Runtime calls get_next() repeatedly to fetch calibration samples.
    Each returned dict must map ONNX input name -> NumPy input batch.
    """
    def __init__(self, dataloader, input_name="input", input_dtype=np.float32):
        self.dataloader = dataloader
        self.input_name = input_name
        self.input_dtype = input_dtype
        self._iterator = iter(self.dataloader)

    def get_next(self):
        try:
            images, _ = next(self._iterator)
            images_np = images.numpy().astype(self.input_dtype)
            return {self.input_name: images_np}
        except StopIteration:
            return None

    def rewind(self):
        self._iterator = iter(self.dataloader)


# Optional preprocessing step before quantization
# In practice this can help, but for some graphs symbolic shape inference can fail.
# If that happens in your environment, skip this block and quantize fp32_onnx_path directly.
print("Running ONNX pre-processing...")
quant_pre_process(
    input_model=fp32_onnx_path,
    output_model_path=preprocessed_onnx_path,
    skip_symbolic_shape=True,
)
print(f"Saved preprocessed ONNX model to: {preprocessed_onnx_path}")

# Build calibration loader
calib_loader = build_calib_loader(
    calib_data_dir,
    batch_size=calib_batch_size,
    subset_size=calib_subset_size,
)

# The ONNX input name should match the exported model input name.
# If your model uses a different input name, replace "input" below.
data_reader = ImageCalibrationDataReader(
    dataloader=calib_loader,
    input_name="input",
    input_dtype=np.float32,
)

print("Running static INT8 quantization with calibration data...")
quantize_static(
    model_input=preprocessed_onnx_path,
    model_output=int8_onnx_path,
    calibration_data_reader=data_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=False,
)

print(f"Saved INT8 quantized ONNX model to: {int8_onnx_path}")

# Size comparison
fp32_size_mb = os.path.getsize(fp32_onnx_path) / (1024 * 1024)
int8_size_mb = os.path.getsize(int8_onnx_path) / (1024 * 1024)

print(f"FP32 ONNX size: {fp32_size_mb:.2f} MB")
print(f"INT8 ONNX size: {int8_size_mb:.2f} MB")
print(f"Size reduction: {(fp32_size_mb - int8_size_mb) / fp32_size_mb * 100:.2f}%")

In [ ]:
import os
import time
import numpy as np
import onnxruntime as ort
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

def build_loader(data_root: str, batch_size: int = 1):
    # Use the same preprocessing as model export / evaluation
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])
    dataset = ImageFolder(root=data_root, transform=transform)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
    )

def evaluate_onnx_cpu(onnx_path, loader, model_label, input_dtype=np.float32):
    # Configure ONNX Runtime for CPU-only inference
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    session = ort.InferenceSession(
        onnx_path,
        sess_options=sess_opts,
        providers=["CUDAExecutionProvider"],
    )
    print(f"{model_label} providers: {session.get_providers()}")

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    correct = 0
    total = 0
    latencies = []
    all_preds = []
    all_labels = []

    for images, labels in loader:
        # ONNX Runtime expects NumPy arrays
        images_np = images.numpy().astype(input_dtype)
        labels_np = labels.numpy()

        start_time = time.perf_counter()
        outputs = session.run([output_name], {input_name: images_np})[0]
        end_time = time.perf_counter()

        latencies.append(end_time - start_time)

        preds = np.argmax(outputs, axis=1)
        all_preds.extend(preds.tolist())
        all_labels.extend(labels_np.tolist())

        correct += (preds == labels_np).sum()
        total += len(labels_np)

    accuracy = 100.0 * correct / total
    total_time = sum(latencies)
    fps = total / total_time if total_time > 0 else float("nan")

    return accuracy, fps, total_time, all_preds, all_labels

# Paths
fp32_model = "../alexnet_mnist.onnx"
int8_model = "../alexnet_mnist_int8_static.onnx"
test_data_dir = "./data/val"
batch_size = 1

# Build evaluation loader
loader = build_loader(test_data_dir, batch_size)
print(f"Test set: {len(loader.dataset)} images  |  batch size: {batch_size}\n")

# Evaluate FP32 ONNX on CPU
acc_fp32, fps_fp32, time_fp32, preds_fp32, labels_fp32 = evaluate_onnx_cpu(
    fp32_model,
    loader,
    "FP32",
    input_dtype=np.float32,
)

# Evaluate INT8 ONNX on CPU
# Calibration-based INT8 ONNX models usually still accept float32 input.
acc_int8, fps_int8, time_int8, preds_int8, labels_int8 = evaluate_onnx_cpu(
    int8_model,
    loader,
    "INT8",
    input_dtype=np.float32,
)

# File sizes
fp32_mb = os.path.getsize(fp32_model) / (1024 * 1024)
int8_mb = os.path.getsize(int8_model) / (1024 * 1024)

# Summary
print("\nSummary")
print(f"{'Metric':<30} {'FP32':>10} {'INT8':>10} {'Delta':>10}")
print("-" * 62)
print(f"{'Accuracy (%)':<30} {acc_fp32:>10.2f} {acc_int8:>10.2f} {acc_int8 - acc_fp32:>+10.2f}")
print(f"{'Throughput (FPS)':<30} {fps_fp32:>10.1f} {fps_int8:>10.1f} {fps_int8 - fps_fp32:>+10.1f}")
print(f"{'Total inference time (s)':<30} {time_fp32:>10.2f} {time_int8:>10.2f} {time_int8 - time_fp32:>+10.2f}")
print(f"{'Model size (MB)':<30} {fp32_mb:>10.2f} {int8_mb:>10.2f} {int8_mb - fp32_mb:>+10.2f}")

speedup = (time_fp32 / time_int8 - 1) * 100 if time_fp32 > 0 and time_int8 > 0 else float("nan")
print(f"\nSpeedup  : {speedup:+.1f}%  ({'faster' if speedup > 0 else 'slower'})")
print(f"Acc drop : {acc_fp32 - acc_int8:.2f} percentage points\n")

print("Sample predictions (INT8 model):")
print(f"  {'True':>6}  {'Pred':>6}  {'Match':>6}")
for true, pred in list(zip(labels_int8, preds_int8))[:10]:
    mark = "✓" if true == pred else "✗"
    print(f"  {true:>6}  {pred:>6}  {mark:>6}")

12. Verify that the exported ONNX model matches its PyTorch counterpart (same inputs, same logits within tolerance).

In [ ]:
# Same random input for both checks: PyTorch logits should match ONNXRuntime for each exported model.
import onnxruntime as ort
import numpy as np

def verify_onnx(torch_model, onnx_path, x, label):
    torch_model.eval()
    with torch.no_grad():
        torch_out = torch_model(x).cpu().numpy()
    sess = ort.InferenceSession(onnx_path)
    in_name = sess.get_inputs()[0].name
    onnx_out = sess.run(None, {in_name: x.cpu().numpy()})[0]
    ok = np.allclose(torch_out, onnx_out, atol=1e-4)
    mad = float(np.max(np.abs(torch_out - onnx_out)))
    print(f"{label}: match={ok}, max abs diff={mad:.6e}")


device = next(finetuned_model.parameters()).device
dummy_input = torch.randn(1, 3, 224, 224, device=device)

verify_onnx(finetuned_model, "../alexnet_mnist.onnx", dummy_input, "Finetuned")

In [ ]:
import onnxruntime as ort
import numpy as np
import time
import os
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

def build_loader(data_root: str, batch_size: int = 1):
    # Build the same preprocessing pipeline used during training/inference.
    # MNIST is grayscale, but AlexNet-style models usually expect 3 channels,
    # so we replicate grayscale into 3 channels.
    transform = transforms.Compose([
        transforms.Resize((224, 224)),                  # Resize to model input size
        transforms.Grayscale(num_output_channels=3),    # Convert 1-channel image to 3 channels
        transforms.ToTensor(),                          # Convert PIL image to tensor in [0, 1]
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),  # Standard ImageNet normalization
    ])

    # ImageFolder expects a folder structure like:
    # data_root/class_0/xxx.png
    # data_root/class_1/yyy.png
    dataset = ImageFolder(root=data_root, transform=transform)

    # shuffle=False keeps evaluation deterministic and aligned with labels
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)


def evaluate_onnx(onnx_path, loader, device_label, input_dtype=np.float32):
    # Create ONNX Runtime session options and enable graph optimizations
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    # Try CUDA first, then fall back to CPU if CUDA is unavailable or unsupported
    session = ort.InferenceSession(
        onnx_path,
        sess_options=sess_opts,
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
    )
    print(f"{device_label} providers: {session.get_providers()}")

    # Read actual input/output names from the ONNX graph
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    # Track classification accuracy, per-batch latency, and all predictions
    correct, total = 0, 0
    latencies = []
    all_preds, all_labels = [], []

    for images, labels in loader:
        # ONNX Runtime takes NumPy arrays as input, not PyTorch tensors
        # input_dtype lets us switch between FP32 and FP16 models
        images_np = images.numpy().astype(input_dtype)
        labels_np = labels.numpy()

        # Measure wall-clock inference time for this batch
        start_time = time.perf_counter()
        outputs = session.run([output_name], {input_name: images_np})[0]
        end_time = time.perf_counter()

        batch_time = end_time - start_time
        latencies.append(batch_time)

        # Take the index of the maximum logit as predicted class
        preds = np.argmax(outputs, axis=1)

        # Save predictions and labels for later inspection
        all_preds.extend(preds.tolist())
        all_labels.extend(labels_np.tolist())

        # Update running accuracy
        correct += (preds == labels_np).sum()
        total += len(labels_np)

    # Final metrics
    accuracy = 100.0 * correct / total
    total_time = sum(latencies)
    fps = total / total_time if total_time > 0 else float("nan")

    return accuracy, fps, total_time, all_preds, all_labels


# Paths to the exported ONNX models
fp32_model = "../alexnet_mnist.onnx"
fp16_model = "../alexnet_mnist_fp16.onnx"

# Validation/test image directory
test_data_dir = "./data/val"

# Batch size 1 is often used here because some exported ONNX models
# may have fixed batch dimension or because we want simple latency testing
batch_size = 1

# Build evaluation loader
loader = build_loader(test_data_dir, batch_size)
print(f"Test set: {len(loader.dataset)} images  |  batch size: {batch_size}\n")

# Evaluate the FP32 ONNX model
acc_fp32, fps_fp32, time_fp32, preds_fp32, labels_fp32 = evaluate_onnx(
    fp32_model, loader, "FP32", input_dtype=np.float32
)

# Evaluate the FP16 ONNX model
# FP16 model expects float16 input if keep_io_types=False was used during conversion
acc_fp16, fps_fp16, time_fp16, preds_fp16, labels_fp16 = evaluate_onnx(
    fp16_model, loader, "FP16", input_dtype=np.float16
)

# Compare model file sizes on disk
fp32_mb = os.path.getsize(fp32_model) / (1024 * 1024)
fp16_mb = os.path.getsize(fp16_model) / (1024 * 1024)

# Print summary table
print("\nSummary")
print(f"{'Metric':<30} {'FP32':>10} {'FP16':>10} {'Delta':>10}")
print("-" * 62)
print(f"{'Accuracy (%)':<30} {acc_fp32:>10.2f} {acc_fp16:>10.2f} {acc_fp16-acc_fp32:>+10.2f}")
print(f"{'Throughput (FPS)':<30} {fps_fp32:>10.1f} {fps_fp16:>10.1f} {fps_fp16-fps_fp32:>+10.1f}")
print(f"{'Total inference time (s)':<30} {time_fp32:>10.2f} {time_fp16:>10.2f} {time_fp16-time_fp32:>+10.2f}")
print(f"{'Model size (MB)':<30} {fp32_mb:>10.2f} {fp16_mb:>10.2f} {fp16_mb-fp32_mb:>+10.2f}")

# Positive value means FP16 is faster; negative means slower
speedup = (time_fp32 / time_fp16 - 1) * 100 if time_fp32 > 0 and time_fp16 > 0 else float("nan")
print(f"\nSpeedup  : {speedup:+.1f}%  ({'faster' if speedup > 0 else 'slower'})")

# Accuracy difference in percentage points
print(f"Acc drop : {acc_fp32 - acc_fp16:.2f} percentage points\n")

# Show a few sample predictions from the FP16 model
print("Sample predictions (FP16 model):")
print(f"  {'True':>6}  {'Pred':>6}  {'Match':>6}")
for true, pred in list(zip(labels_fp16, preds_fp16))[:10]:
    mark = '✓' if true == pred else '✗'
    print(f"  {true:>6}  {pred:>6}  {mark:>6}")

## 12. Recap

In this workshop you adapted pretrained AlexNet to MNIST, finetuned the classifier, compared accuracy and throughput against a pruned variant, exported both networks to ONNX, and checked that ONNX Runtime matches PyTorch on the same random input.

**Exported models** (paths follow your export cells, e.g. `../` relative to this notebook):

- **`alexnet_mnist.onnx`** — finetuned model weights.
- **`alexnet_mnist_quantized.onnx`** — quantized finetuned model weights; use as the default deployment artifact.

You can run these in ONNX Runtime, TensorRT, OpenVINO, or any runtime that supports the ONNX opset you exported with.
